In [ ]:
! nvidia-smi

In [ ]:
from datasets import load_dataset
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import gc
import time
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig,SFTTrainer
from PIL import Image
from qwen_vl_utils import process_vision_info

In [ ]:
from huggingface_hub import HfFolder, whoami

#Replace 'your_token' with your actual Hugging Face token
hf_token = 'YOUR_HF_TOKEN'  #Write token for Granite-Vision-3.2 Model
HfFolder.save_token(hf_token)

user = whoami()
print(f"Logged in as {user['name']}")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
system_message = """You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.

For circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.

Focus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy."""

In [ ]:
def load_image(image_path, target_size=(512, 512)):
    with Image.open(image_path) as img:
        img = img.convert("RGB")  # Ensure compatibility
        img = img.resize(target_size, Image.LANCZOS) #High-Quality Downscaling
        return img.copy()

def format_data(sample):
    return [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": load_image(sample["image"]),
                },
                {
                    "type": "text",
                    "text": sample["conversations"][0]["value"],
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": sample["conversations"][1]["value"]
                }
            ],
        },
    ]

In [ ]:
raw_datasets = load_dataset("json", data_files="json_data_12_03_2025_final.json")
split_datasets = raw_datasets["train"].train_test_split(test_size=0.2)
train_dataset = split_datasets["train"] 
eval_dataset = split_datasets["test"]

print(f"Training Dataset Size: {len(train_dataset)}") 
print(f"Evaluation Dataset Size: {len(eval_dataset)}")

In [ ]:
train_dataset[0]

In [ ]:
train_dataset_formatted = [format_data(sample) for sample in train_dataset]
eval_dataset_formatted = [format_data(sample) for sample in eval_dataset]

In [ ]:
train_dataset_formatted[0]

In [ ]:
# Save train_dataset and eval_dataset to disk
train_dataset_path = "./datasets/train_dataset_13_03_2025_qwen"
eval_dataset_path = "./datasets/eval_dataset_13_03_2025_qwen"

train_dataset.save_to_disk(train_dataset_path)
eval_dataset.save_to_disk(eval_dataset_path)

print(f"Train dataset saved to: {train_dataset_path}")
print(f"Eval dataset saved to: {eval_dataset_path}")

In [ ]:
#Remove Model and Clean GPU

def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory() #Clearing variable, model and GPU

In [ ]:
model_id = "Qwen/Qwen2.5-VL-7B-Instruct" #"HuggingFaceTB/SmolVLM-500M-Instruct"

In [ ]:
# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

# Apply PEFT model adaptation
peft_model = get_peft_model(model, peft_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

In [ ]:
# Configure training arguments
training_args = SFTConfig(
    output_dir="qwen2.5-vl-instruct-7b-13-03-2025",  # Directory to save the model
    num_train_epochs=1,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size for training
    gradient_accumulation_steps=8,  # Steps to accumulate gradients
    remove_unused_columns = False,  # Keep unused columns in dataset
    
    # Optimizer and scheduler settings
    optim="adamw_torch_fused",  # Optimizer type
    learning_rate=2e-4,  # Learning rate for training
    lr_scheduler_type="constant",  # Type of learning rate scheduler
    
    # Logging and evaluation
    logging_steps=10,  # Steps interval for logging
    #eval_steps=100,  # Steps interval for evaluation
    #eval_strategy="steps",  # Strategy for evaluation
    save_strategy="steps",  # Strategy for saving the model
    save_steps=25,  # Steps interval for saving
    save_total_limit=10,
    metric_for_best_model="eval_loss",  # Metric to evaluate the best model
    greater_is_better=False,  # Whether higher metric values are better
    #load_best_model_at_end=True,  # Load the best model after training
    
    # Mixed precision and gradient settings
    bf16=True,  # Use bfloat16 precision
    max_grad_norm=0.3,  # Maximum norm for gradient clipping
    warmup_ratio=0.03,  # Ratio of total steps for warmup
    
    # Hub and reporting
    push_to_hub=True,  # Whether to push model to Hugging Face Hub
    report_to="tensorboard",  # Reporting tool for tracking metrics
    logging_dir="qwen2.5-vl-instruct-7b-13-03-2025/logs",
    
    # Gradient checkpointing settings
    gradient_checkpointing=True,  # Enable gradient checkpointing for memory efficiency
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Options for gradient checkpointing
    # Dataset configuration
    dataset_text_field="",  # Text field in dataset
    dataset_kwargs={"skip_prepare_dataset": True},  # Additional dataset options
    #max_seq_length=1024  # Maximum sequence length for input
)


In [ ]:
# Create a data collator to encode text and image pairs
def collate_fn(examples):
    # Get the texts and images, and apply the chat template
    texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]  # Prepare texts for processing
    image_inputs = [process_vision_info(example)[0] for example in examples]  # Process the images to extract inputs

    # Tokenize the texts and process the images
    batch = processor(text=texts, images=image_inputs, return_tensors="pt", padding=True)  # Encode texts and images into tensors

    # The labels are the input_ids, and we mask the padding tokens in the loss computation
    labels = batch["input_ids"].clone()  # Clone input IDs for labels
    labels[labels == processor.tokenizer.pad_token_id] = -100  # Mask padding tokens in labels

    # Ignore the image token index in the loss computation (model specific)
    image_tokens = [processor.tokenizer.convert_tokens_to_ids(processor.image_token)]  # Convert image token to ID

    # Mask image token IDs in the labels
    for image_token_id in image_tokens:
        labels[labels == image_token_id] = -100  # Mask image token IDs in labels

    batch["labels"] = labels  # Add labels to the batch

    return batch  # Return the prepared batch

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    data_collator=collate_fn,
    peft_config=peft_config,
    tokenizer=processor.tokenizer,
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir qwen2.5-vl-instruct-7b-13-03-2025/logs --port=6008

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(training_args.output_dir)

In [ ]:
trainer.push_to_hub()

In [ ]:
clear_memory()